Glad you are interested to know the details about how we trained a neural network. This will only give a high level understanding for your curiousity. The specific architectural, optimisation details are not necessary. 

For any deep learning application, you first start with the data. What does the data look like in our case? 

Think about it for a second. Have a look at an empty TicTacToe board. 

In [ ]:

import os 
import sys 
import numpy as np 
main_dir = os.path.dirname(os.getcwd())
sys.path.append(main_dir)
from src.game_utils import TicTacToeGame, map_index_to_move


game = TicTacToeGame()
game.show_game_state()

Building a dataset requires making choices. Sometimes, these choices are made implicitly and that can have consequences for the performance of the model. What we describe here is a set of choice keeping in mind the requirements of a course and simplicity. Keep this in mind while you read the rest, and formulate your own ideas about what choices you would make instead and what consequences that would have. 

The first choice was to select the input and the output for the model. As input, we provide information about the current state of the board and the output (or the target) is selected as the next move that is made. That is, our deep learning model is answering the question: "Given the state of the board what would be the next move to make?". What information do you think we are throwing away? 

Now for the practical questions. How can we describe the current state of the board? Suppose the board looks like: 
> [[ ] [ ] [O]]

> [[ ] [X] [ ]]

> [[ ] [ ] [O]]

What we have here is two positions marked by the "O" and one by the "X". We can represent the letters with integers. We choose "X" to be the first player so it is represented by +1 and "O" to be the second player so it has -1. So our grid becomes
> [[ ] [ ] [-1]]

> [[ ] [1] [  ]]

> [[ ] [ ] [-1]]

While we can keep the grid 2d, it is simpler to build a model with a 1d vector. So we just create a 1d vector with length 9 so that each of the position in that cell corresponds to the 2d grid through the numpy reshape function. So, 
> [[ ] [ ] [O]]           [[ ] [ ] [-1]]

> [[ ] [X] [ ]]   -->     [[ ] [1] [  ]]   -->    [0, 0, -1, 0, 1, 0, 0, 0, -1]

> [[ ] [ ] [O]]           [[ ] [ ] [-1]]

So empty spaces are filled with 0 and X and O are filled with +1 and -1 respectively. 

In [ ]:
game.show_game_state(use_game_state=np.array([0, 0, -1, 0, 1, 0, 0, 0, -1]))

So our input corresponds to a 1d vector of size 9. For output, we selected just a scalar number which points to the index where the next move is made. 

If input is: [0, 0, -1, 0, 1, 0, 0, 0, -1], the next move for X could correspond to location B3 on the grid, which is the index location 5. 


In [ ]:
print(f"Placing X at position 5 or B3")
game.show_game_state(use_game_state=np.array([0, 0, -1, 0, 1, 1, 0, 0, -1]))

Alright, now we have a way to extract game state and the next move. The goal now is to train a model which can learn the legal moves. By this, we mean just that the model should not place a mark where there is already another mark. For this, we generate several "simulated" games. In this simulated game, the next move is picked from any one of the available positions in the grid (which corresponds to 0 in the 1d representation). The core strategy in the game of Tic Tac Toe is about picking the next move. In reality, when we play games that strategy depends on several factors like the likely next move by the opponent. However, to make our life easy we just pick a spot at random :) 

This allows a simple way to generate hundreds of target positions for given board states allowing us to train a model. Click the next cell to generate this data. 

In [ ]:
# First step, generate training data using random game
from tqdm import tqdm
num_games_training = 5000 
training_data = []
for _ in tqdm(range(num_games_training), desc="Generating training data"):
    game = TicTacToeGame()
    game_data = game.play(visualise=False)
    training_data.append(game_data['all_user_inputs'][1:]) # Exclude the first move which is always the same


Great! Now that we have our training data, let us have a look at it. Click the next cell to take a peek.

In [ ]:
# Show how the training data looks like for one game
import pandas as pd
from tabulate import tabulate
print("Example game data:")
df = pd.DataFrame(training_data[0], columns=['board_state', 'player', 'next_move'])
print(tabulate(df, headers='keys', tablefmt='psql'))



What we then need to do is use this training data to, well, train a model. We will not go into this yet. The two functions 'define_model()' and 'train_model()' takes care of this. Run the next code cell to proceed with the training. 

In [ ]:
# What we need for training data is the current board state as input and the next move as the target
from src.train_game_utils import define_model, train_model
tictactoe_model = define_model(reproducible=True)
# Prepare the training data
X_train = []
y_train = []
for game in training_data:
    for move in game:
        board_state, player, next_move = move
        X_train.append(board_state)
        y_train.append(next_move)

# Train the model
tictactoe_model = train_model(tictactoe_model, X_train, y_train, num_epochs=100)


Did the model learn anything? To test it, we have to just play a few games and see whethen the model predicts any illegal moves. 

In [ ]:
# Did the model learn anything? Let's test it against a random player to see if it predicts an illegal move
test_game = TicTacToeGame(player_1=tictactoe_model)
result = test_game.play(visualise=False)
# Show any illegal moves made
if len(result["illegal_moves"]) > 0:
    print(f"{len(result['illegal_moves'])} illegal moves made by the model:")
    for i, move in enumerate(result["illegal_moves"]):
        print(f"Move {i+1}: Model chose {map_index_to_move(move[2])}")
        test_game.show_game_state(use_game_state=move[0])
else:
    print("No illegal moves made by the model!")



Was it all legal moves? Check automatically for a large number of games and see what percentage of games were legal. 

In [ ]:
# Test it for many games to see how it performs
number_of_games_with_illegal_moves = 0
num_test_games = 1000
for _ in tqdm(range(num_test_games), desc="Testing model for illegal moves"):
    game_data = TicTacToeGame(player_1="/Users/alokbharadwaj/dev/ai4nanobiology/week_1/src/models/Bob.pt")
    game_data = game_data.play(visualise=False)
    if game_data['illegal_moves']:
        number_of_games_with_illegal_moves += 1

print(f"Model made illegal moves in {number_of_games_with_illegal_moves} out of {num_test_games} games.")
print(f"Percentage of games with illegal moves: {number_of_games_with_illegal_moves / num_test_games * 100:.2f}%")

Not happy with the training? Increase the number of training data, or the number of epochs and see if that helps you get a better model. 

In [ ]:
# Save the model to a file so we can load it later without retraining
save = False
if save:
    import torch
    import datetime
    model_save_folder = os.path.join(main_dir, 'src', 'models')
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    model_save_path = os.path.join(model_save_folder, f'tictactoe_model_{timestamp}.pt')
    torch.save(tictactoe_model.state_dict(), model_save_path)
    print(f"Model saved to {model_save_path}")